<a href="https://colab.research.google.com/github/pyypyyy/Asiamiestutkinto-trainer/blob/main/Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klikkaa joka solusta nuolta

In [ ]:
# @title
!pip install openai rich

!pip -q install openai rich ipywidgets

from openai import OpenAI
from rich.console import Console
from rich.table import Table
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets
import json, os, uuid
from datetime import datetime

# Lisää tähän API-Key

In [ ]:
# @title
from getpass import getpass
API_KEY = getpass("Paste your OpenAI API key (hidden): ")

In [12]:
# @title

MODEL = "gpt-5"

client = OpenAI(api_key=API_KEY)
console = Console()

def load_items(path: str = "items.jsonl"):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Ei löytynyt {path}")
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

def load_prompt_template(path: str = "prompt_template.txt"):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Ei löytynyt {path}")
    return p.read_text(encoding="utf-8")

def build_messages(template: str, *, question_text: str, rubric: dict, user_answer: str, max_points: int):
    user_prompt = (template
                   .replace("{{question_text}}", question_text)
                   .replace("{{rubric_json}}", json.dumps(rubric, ensure_ascii=False))
                   .replace("{{user_answer}}", user_answer)
                   .replace("{{max_points}}", str(max_points)))
    return [
        {"role": "system", "content": "Olet täsmällinen, tiukka arvioija. Pisteytä vain RUBRICin mukaan ja palauta JSON."},
        {"role": "user", "content": user_prompt},
    ]

def ensure_json(text: str):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end+1])
        raise

def pretty_print_result(result: dict, max_points: int):
    fp = result.get("final_points", result.get("total_points", 0))
    mp = result.get("max_points", max_points)
    print(f"\nPisteet: {fp}/{mp}")
    if "short_feedback" in result:
        print("Palaute:", result["short_feedback"])
    # Taulukot (valinnaisesti)
    if "criteria" in result:
        t = Table(title="Kriteerit")
        t.add_column("ID", style="cyan"); t.add_column("Täyttyi", style="green")
        t.add_column("Pisteet", justify="right"); t.add_column("Perustelu", overflow="fold")
        for c in result["criteria"]:
            t.add_row(c.get("id","-"), "✅" if c.get("met") else "❌", str(c.get("points",0)), c.get("explanation",""))
        console.print(t)
    if result.get("penalties_applied"):
        t2 = Table(title="Vähennykset")
        t2.add_column("ID", style="red"); t2.add_column("Vähennys", justify="right"); t2.add_column("Perustelu", overflow="fold")
        for p in result["penalties_applied"]:
            t2.add_row(p.get("id","-"), str(p.get("points",0)), p.get("explanation",""))
        console.print(t2)

def grade_once(item, user_answer: str):
    template = load_prompt_template("prompt_template.txt")
    messages = build_messages(template,
                              question_text=item["question_text"],
                              rubric=item["rubric"],
                              user_answer=user_answer,
                              max_points=item.get("max_points", 50))
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    raw = resp.choices[0].message.content.strip()
    result = ensure_json(raw)
    pretty_print_result(result, item.get("max_points", 50))
    return result

import random
items = load_items("items.jsonl")

current_item = [random.choice(items)]  # lista, jotta referenssiä voi vaihtaa

def show_item(it):
    print(f"{it['title']} — {it['year']} {it['exam']} {it['section']}")
    print(it["question_text"])

# Harjoitustehtävä

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display

btn_new = widgets.Button(description="Arvo uusi tehtävä")
out_question = widgets.Output()

def on_new_click(_):
    with out_question:
        out_question.clear_output()
        it = random.choice(items)
        current_item[0] = it           # <-- tärkeää!
        show_item(it)

btn_new.on_click(on_new_click)

with out_question:
    show_item(current_item[0])

display(btn_new, out_question)

ta = widgets.Textarea(layout=widgets.Layout(width='100%', height='200px'))
btn_grade = widgets.Button(description="Arvioi", button_style='success')
out_grade = widgets.Output()

def on_grade_click(btn):
    with out_grade:
        out_grade.clear_output()
        if not ta.value.strip():
            print("Anna vastaus ensin.")
            return
        it = current_item[0]  # <-- käytä tätä!
        print(f"[DEBUG] Grading item: {it['id']} — {it['title']}")
        res = grade_once(it, ta.value)  # grade_once lukee rubricin suoraan it:stä

btn_grade.on_click(on_grade_click)

display(ta, btn_grade, out_grade)